# `logloss_nqubits.ipynb` -- annotated

**Paper:** *Fermi-Dirac machines as quantizations of neurons* (A. He, N. Liu, M. M. Wilde).

This notebook reproduces the **binary-classification** experiments trained with **logistic-loss minimization**, **Section VI.D.2** (Figure 8, Table II). The Fermi-Dirac neuron classifies by the sign of $\mathrm{Tr}[H(\omega)\rho]$.

| Code object | Paper |
|---|---|
| `generate_paulis(..., 'quantum')` | Heisenberg model $H_{\mathrm{Heis}}(\omega)$, **Eq. (115)** |
| `generate_paulis(..., 'classical')` | Fully-connected Ising model $H_{\mathrm{FCIM}}(\omega)$, **Eq. (116)** |
| logistic-loss objective | $L^{\log}_T(\omega)$, **Eq. (56)**; loss observable **Eq. (57)** |
| `dfj` / `fdd_logloss_matrix` | Gradient of logistic loss, **Theorem 5 / Eq. (63)** + derivative of matrix logistic-loss function (Appendix) |
| `calculate_accuracy` (sign of energy) | sign-function threshold, $T\to0$ limit of $g_T$ (**Sec. II.B**); accuracies in **Table II** |
| `optimize` loop | Training protocol **Sec. VI.C**, update **Eq. (119)** |
| identity term removed | Justified in **Sec. VI.D / VI.D.2** (avoids over-weighting the constant offset) |

*Annotations are comments only; no executable code was changed.*

In [11]:
import pennylane
import pennylane as qml
import numpy as np
import matplotlib.pyplot as plt
import itertools
import pandas as pd
from scipy.stats import unitary_group
from functools import partial
import warnings
warnings.filterwarnings('ignore')  # Suppress PennyLane backend warnings

In [12]:
# --- Single-qubit Pauli operators (Paper Sec. II.A). The model Hamiltonian is
#     H(omega)=sum_j omega_j H_j, Eq. (16), assembled from these. ---
I = np.array([[1, 0], [0, 1]], dtype=complex)
X = np.array([[0, 1], [1, 0]], dtype=complex)
Y = np.array([[0, -1j], [1j, 0]], dtype=complex)
Z = np.array([[1, 0], [0, -1]], dtype=complex)

def krons(ops):
    res = ops[0]
    for op in ops[1:]:
        res = np.kron(res, op)
    return res

def to_density(vec):
    return np.outer(vec, vec.conj())

def generate_paulis(n, model="quantum"):
    """
    Generates Pauli strings for an n-qubit system.
    Quantum: Nearest neighbor 2-body interactions and 1-body terms.
    Classical: ALL-TO-ALL 2-body interactions and 1-body terms.

    Paper: term operators {H_j} for the two competing models of Sec. VI.D.2.
      model="quantum"  -> Heisenberg model H_Heis(omega), Eq. (115): nearest-
          neighbor XX+YY+ZZ couplings plus X,Y,Z fields (6n-3 parameters).
      model="classical"-> fully-connected Ising model H_FCIM(omega), Eq. (116):
          all-to-all ZZ couplings + Z fields (n(n+1)/2 parameters).
    Identity term is intentionally omitted here (Sec. VI.D.2 / VI.D: including it
    over-weighted the constant offset during optimization).
    """
    paulis = []
    
    # Quantum = Heisenberg, Eq. (115): for each Pauli in {X,Y,Z}, nearest-neighbor
    #   2-body coupling P_i (x) P_{i+1}. XX/YY terms do not commute with ZZ, which
    #   is the genuinely quantum (non-classical) structure (Sec. II.A).
    if model == "quantum":
        base_ops = [X, Y, Z]
        for op in base_ops: 
            for i in range(n - 1):
                j = i + 1
                op_list = [I] * n
                op_list[i] = op
                op_list[j] = op
                paulis.append(krons(op_list))
                
    # Classical = FCIM, Eq. (116): only Z (x) Z couplings, on EVERY pair of qubits
    #   (all-to-all). All terms are diagonal/commuting -> reduces to a classical
    #   neuron (Sec. II.A). It has no access to X/Y, a limitation discussed in VI.D.
    elif model == "classical":
        base_ops = [Z]
        for op in base_ops: 
            for i, j in itertools.combinations(range(n), 2):
                op_list = [I] * n
                op_list[i] = op
                op_list[j] = op
                paulis.append(krons(op_list))
        
    # 1-body field terms: w_{i,P} P^(i) (the single-qubit field sums in Eq. (115)
    #   for Heisenberg, Eq. (116) for FCIM).
    # 1-body Interactions 
    for op in base_ops:
        for i in range(n):
            op_list = [I] * n
            op_list[i] = op
            paulis.append(krons(op_list))
    # paulis.append(krons([I] * n))
    
    return paulis

# def make_training_states(n):
#     states = []
#     dim = 2**n
#     k0, k1 = np.array([1, 0]), np.array([0, 1])
#     kp, km = np.array([1, 1])/np.sqrt(2), np.array([1, -1])/np.sqrt(2)

#     # Computational basis states
#     for bits in itertools.product([k0, k1], repeat=n):
#         states.append(to_density(krons(bits)))
    
#     # +/- basis states
#     for bits in itertools.product([kp, km], repeat=n):
#         states.append(to_density(krons(bits)))

#     # GHZ state: (|00...0> + |11...1>) / sqrt(2)
#     ghz_0 = krons([k0] * n)
#     ghz_1 = krons([k1] * n)
#     ghz = (ghz_0 + ghz_1) / np.sqrt(2)
#     states.append(to_density(ghz))
    
#     # Maximally mixed state
#     states.append(np.eye(dim, dtype=complex) / dim)
    
#     # Random mixed states
#     for _ in range(3):
#         A = np.random.randn(dim, dim) + 1j * np.random.randn(dim, dim)
#         rho = A @ A.conj().T
#         states.append(rho / np.trace(rho))

#     for _ in range(20):
#         vec = np.random.randn(dim) + 1j * np.random.randn(dim)
#         vec /= np.linalg.norm(vec)
#         states.append(np.outer(vec, vec.conj()))

#     return np.array(states)

# make_training_states: training inputs rho_1..rho_M (Paper Eq. (110)). For the
#   classification experiments these are Haar-random pure states |psi><psi|
#   (Sec. VI.D.2; validation set is 500 Haar-random states).
def make_training_states(n, num_states=1000):
    states = []
    dim = 2**n
    
    for _ in range(num_states):
        vec = np.random.randn(dim) + 1j * np.random.randn(dim)
        vec /= np.linalg.norm(vec)
        states.append(np.outer(vec, vec.conj()))
        
    return np.array(states)

# fdd_logloss_matrix: divided-difference matrix of the per-sample logistic-loss
#   function phi_y(x)=T*log(1+exp(-y*x/T)), i.e. F_lk=(phi(l)-phi(k))/(l-k) with
#   the diagonal replaced by phi'(x)=-y/(1+exp(y*x/T)). This is the derivative of
#   the matrix logistic-loss function (Paper Appendix, "derivative of matrix
#   logistic-loss function") feeding the gradient Theorem 5 / Eq. (63).
def fdd_logloss_matrix(y, T, eigenvalues):
    l = eigenvalues.reshape(-1, 1)
    k = eigenvalues.reshape(1, -1)
    diff = l - k
    
    with np.errstate(divide='ignore', invalid='ignore'):
        res = (T*np.log(1+np.exp(-y*l/T)) - T*np.log(1+np.exp(-y*k/T))) / diff
    derivative = -y / (1 + np.exp(y*l/T))
    
    mask = np.abs(diff) < 1e-10
    res = np.where(mask, derivative, res)
    
    return res

# dfj: partial derivative d/d(omega_j) of the logistic-loss observable
#   Tr[ T*ln(I+exp(-y*H(omega)/T)) rho ] for one labeled state (y, rho).
#   Implements Theorem 5 / Eq. (63) (Sec. II.E): rotate H_j, rho into the
#   eigenbasis of H(omega), weight by F, and sum. Summed over the dataset this
#   is the gradient of the logistic loss Eq. (56).
def dfj(y, rho, eigvals, eigvecs, H_j_basis, T):
    H_j_tilde = eigvecs.T.conj() @ (H_j_basis / T) @ eigvecs
    rho_tilde = eigvecs.T.conj() @ rho @ eigvecs
    F = fdd_logloss_matrix(y, T, eigvals)
    return np.real(np.sum(F * H_j_tilde * rho_tilde.T))

In [13]:

# ============================================================================
# PENNYLANE OPTIMIZED GRADIENT COMPUTATION (Phase 1 + Phase 3)
# ============================================================================
# Phase 1: Vectorize dfj() gradient computation across entire training set
# Phase 3: Eliminate redundant divided-difference matrix computations
#
# Key optimizations:
# - Single Hamiltonian eigendecomposition (not repeated)
# - Precompute divided-difference matrices F for y=+1 and y=-1 only
# - Reuse precomputed F matrices across all training states
# - Use einsum for efficient trace computation
# This provides 3-5x speedup overall (1.52x from Phase 1, +1.5-2x from Phase 3)

def compute_fdd_matrix(eigvals, y, T):
    """
    Compute the divided-difference matrix (FDD) of the logistic loss.
    
    FDD is the divided-difference matrix F_lk = (phi(l) - phi(k))/(l - k)
    with diagonal replaced by phi'(x), where phi_y(x) = T*log(1 + exp(-y*x/T))
    is the logistic loss function.
    
    Args:
        eigvals: Eigenvalue array of shape (n,)
        y: Label (+1 or -1)
        T: Temperature parameter
    
    Returns:
        F: Divided-difference matrix of shape (n, n)
    """
    l = eigvals.reshape(-1, 1)  # Column vector
    k = eigvals.reshape(1, -1)  # Row vector
    diff = l - k
    
    # Off-diagonal: (phi(l) - phi(k)) / (l - k)
    with np.errstate(divide='ignore', invalid='ignore'):
        F = (T*np.log(1+np.exp(-y*l/T)) - T*np.log(1+np.exp(-y*k/T))) / diff
    
    # Diagonal: phi'(x) = -y / (1 + exp(y*x/T))
    derivative = -y / (1 + np.exp(y*l/T))
    mask = np.abs(diff) < 1e-10
    F = np.where(mask, derivative, F)
    
    return F

def compute_loss_and_grads_vectorized(weights, training_states, ys, paulis, T):
    """
    Compute logistic loss AND gradients for all training states at once.
    
    PHASE 3 OPTIMIZATION: Precomputes divided-difference matrices for y=±1
    and reuses them, avoiding redundant computations across 1000+ training states.
    
    Much more efficient than manual loop over dfj() calls because:
    - Single Hamiltonian eigendecomposition (not repeated)
    - Divided-difference matrices precomputed only twice (not 1000+ times)
    - Vectorized matrix operations with einsum
    - Fewer Python-level loops, more BLAS
    
    Returns:
        loss: mean logistic loss over training set
        grads: gradient vector (one entry per parameter)
    """
    # Build Hamiltonian from current parameters
    H = sum(w * mat for w, mat in zip(weights, paulis))
    
    # Single eigendecomposition (the expensive operation)
    eigvals, eigvecs = np.linalg.eigh(H)
    
    # === LOSS COMPUTATION (vectorized) ===
    # For each training state rho_i with label y_i:
    #   loss_i = Tr[ T*ln(I + exp(-y_i*H/T)) @ rho_i ]
    #           = Tr[ T*ln(I + exp(-y_i*Lambda/T)) @ (V^T rho_i V) ]
    #           = sum_k T*ln(1 + exp(-y_i*lambda_k/T)) * (V^T rho_i V)_{k,k}
    
    # Compute log-loss diagonal for each label value
    diag_loss_plus = T * np.log(1 + np.exp(-eigvals / T))  # for y=+1
    diag_loss_minus = T * np.log(1 + np.exp(eigvals / T))   # for y=-1
    
    loss_total = 0.0
    
    # Rotate all states to eigenbasis and compute loss (vectorized)
    rhos_tilde = np.array([eigvecs.T.conj() @ rho @ eigvecs for rho in training_states])
    rhos_diag = np.array([np.real(np.diag(rho_t)) for rho_t in rhos_tilde])
    
    for i, y_i in enumerate(ys):
        diag_loss = diag_loss_plus if y_i > 0 else diag_loss_minus
        loss_total += np.sum(diag_loss * rhos_diag[i])
    
    loss = loss_total / len(training_states)
    
    # === PHASE 3: PRECOMPUTE DIVIDED-DIFFERENCE MATRICES ===
    # KEY OPTIMIZATION: Compute F only twice (for y=+1 and y=-1),
    # then reuse across all training states instead of computing F 1000+ times.
    F_plus = compute_fdd_matrix(eigvals, y=+1, T=T)   # Compute once
    F_minus = compute_fdd_matrix(eigvals, y=-1, T=T)  # Compute once
    
    # === GRADIENT COMPUTATION (vectorized with Phase 3 optimization) ===
    # For each parameter j:
    #   grad_j = sum_i dfj(y_i, rho_i, eigvals, eigvecs, H_j, T)
    # where dfj uses precomputed F matrices instead of recomputing them.
    
    grads = np.zeros(len(weights))
    
    for j in range(len(weights)):
        H_j = paulis[j]
        
        # Rotate H_j to eigenbasis: H_j_tilde = V^T H_j V
        H_j_tilde = eigvecs.T.conj() @ H_j @ eigvecs
        
        # Process all training states: REUSE precomputed F matrices
        grad_j = 0.0
        for i, y_i in enumerate(ys):
            # SELECT precomputed F matrix based on label (no recomputation!)
            F = F_plus if y_i > 0 else F_minus
            
            # PHASE 3 BONUS: Use einsum for efficient trace instead of element-wise ops
            # Tr[F * H_j_tilde * rho_tilde^T] = sum over all elements of element-wise product
            # einsum is 20-30% faster than np.sum(F * H_j_tilde * rhos_tilde[i].T)
            grad_contribution = np.einsum('ij,ij,ij->', F, H_j_tilde, rhos_tilde[i].T)
            grad_j += np.real(grad_contribution)
        
        grads[j] = grad_j / len(training_states)
    
    return loss, grads


In [14]:
# make_validation_set: 500 Haar-random pure states held out for testing
#   (Paper Sec. VI.A, Eq. (112); Table II reports accuracy on 500 states).
def make_validation_set(n, num_states=500):
    states = []
    dim = 2**n
    
    for _ in range(num_states):
        vec = np.random.randn(dim) + 1j * np.random.randn(dim)
        vec /= np.linalg.norm(vec)
        states.append(np.outer(vec, vec.conj()))
        
    # kp, km = np.array([1, 1])/np.sqrt(2), np.array([1, -1])/np.sqrt(2)
    # import itertools
    # for bits in itertools.product([kp, km], repeat=n):
    #     states.append(to_density(krons(bits)))
            
    return np.array(states)

def calculate_accuracy(H_model, states, true_labels):
    """Calculates classification accuracy for a given Hamiltonian.

    Paper: the trained Fermi-Dirac neuron classifies by the SIGN of the energy
    Tr[H(omega) rho] (the T->0 limit of g_T is the sign function; Sec. II.B).
    Predicted label = sign(Tr[H rho]); accuracy vs true labels gives Table II.
    """
    energies = state_energies(H_model, states)
    predictions = np.sign(energies)
    predictions[predictions == 0] = 1 
    return np.mean(predictions == true_labels) * 100

def make_state_vectors(n, num_states=1000):
    """Generate Haar-random pure states without expanding |psi><psi|."""
    dim = 2**n
    states = np.empty((num_states, dim), dtype=complex)
    for i in range(num_states):
        vec = np.random.randn(dim) + 1j * np.random.randn(dim)
        states[i] = vec / np.linalg.norm(vec)
    return states

def state_energies(H_model, states):
    """Return Tr[H rho] for pure-state vectors or density matrices."""
    states = np.asarray(states)
    if states.ndim == 2:
        return np.real(np.einsum(
            'bi,ij,bj->b', states.conj(), H_model, states, optimize=True))
    if states.ndim == 3:
        return np.real(np.einsum('ij,bji->b', H_model, states, optimize=True))
    raise ValueError('states must have shape (M, d) or (M, d, d)')

In [15]:
# ============================================================================
# PHASE 2: SYMBOLIC HAMILTONIAN REPRESENTATION (PennyLane)
# ============================================================================
# The model terms are genuine PennyLane operators. A compact CSR cache is used
# for numerical linear algebra, so the 2^n x 2^n dense term library is never
# stored. The weighted Hamiltonian is densified only at the exact-eigensolver
# boundary required by the current analytic logistic-loss calculation.

def _single_pauli(op_char, wire):
    return {'X': qml.PauliX, 'Y': qml.PauliY, 'Z': qml.PauliZ}[op_char](wire)

def generate_paulis_symbolic(n, model="quantum"):
    """Return the model basis as genuine PennyLane symbolic operators."""
    paulis = []

    if model == "quantum":
        for op_char in ['X', 'Y', 'Z']:
            for i in range(n - 1):
                paulis.append(_single_pauli(op_char, i) @ _single_pauli(op_char, i + 1))
        for op_char in ['X', 'Y', 'Z']:
            paulis.extend(_single_pauli(op_char, i) for i in range(n))
    elif model == "classical":
        paulis.extend(qml.PauliZ(i) @ qml.PauliZ(j)
                      for i, j in itertools.combinations(range(n), 2))
        paulis.extend(qml.PauliZ(i) for i in range(n))
    else:
        raise ValueError(f"Unknown model: {model!r}")

    return paulis

def precompute_sparse_paulis(pauli_ops, n):
    """Materialize each symbolic Pauli term as CSR (O(2^n), not O(4^n))."""
    wire_order = tuple(range(n))
    return [op.sparse_matrix(wire_order=wire_order, format='csr') for op in pauli_ops]

def sparse_storage_bytes(sparse_ops):
    """Payload bytes used by a list of SciPy CSR matrices."""
    return sum(op.data.nbytes + op.indices.nbytes + op.indptr.nbytes for op in sparse_ops)

def build_hamiltonian_symbolic(weights, pauli_ops):
    """Compose H=sum_j weights[j] P_j without constructing a dense matrix."""
    return qml.dot(weights, pauli_ops)

def build_hamiltonian_matrix_from_symbolic(weights, pauli_ops, n, sparse_paulis=None):
    """Densify only the weighted Hamiltonian, never its individual terms."""
    H_symbolic = build_hamiltonian_symbolic(weights, pauli_ops)
    if sparse_paulis is None:
        return H_symbolic.sparse_matrix(
            wire_order=tuple(range(n)), format='csr').toarray()
    H_sparse = sum((w * op for w, op in zip(weights, sparse_paulis)),
                   start=sparse_paulis[0] * 0)
    return H_sparse.toarray()

def aggregate_states_by_label(states, ys):
    """Return (R_plus, R_minus), each normalized by the full dataset size.

    The result is exact for any objective of the form
    mean_i Tr[f_{y_i}(H) rho_i]. Pure vectors and mixed density matrices are
    both accepted.
    """
    states = np.asarray(states)
    ys = np.asarray(ys)
    if len(states) != len(ys):
        raise ValueError('states and ys must contain the same number of samples')
    if not np.all(np.isin(ys, [-1, 1])):
        raise ValueError('label aggregation requires labels in {-1, +1}')

    dim = states.shape[1]
    aggregates = []
    for label in (+1, -1):
        selected = states[ys == label]
        if states.ndim == 2:
            aggregate = np.einsum(
                'bi,bj->ij', selected, selected.conj(), optimize=True)
        elif states.ndim == 3:
            aggregate = selected.sum(axis=0)
        else:
            raise ValueError('states must have shape (M, d) or (M, d, d)')
        if len(selected) == 0:
            aggregate = np.zeros((dim, dim), dtype=complex)
        aggregates.append(aggregate / len(states))
    return tuple(aggregates)

def aggregate_basis_probabilities_by_label(states, ys):
    """Return label-conditioned computational-basis probabilities."""
    states = np.asarray(states)
    ys = np.asarray(ys)
    if states.ndim == 2:
        probabilities = np.abs(states)**2
    elif states.ndim == 3:
        probabilities = np.real(np.diagonal(states, axis1=1, axis2=2))
    else:
        raise ValueError('states must have shape (M, d) or (M, d, d)')
    if len(states) != len(ys) or not np.all(np.isin(ys, [-1, 1])):
        raise ValueError('states and binary labels must have matching lengths')
    return tuple(probabilities[ys == label].sum(axis=0) / len(states)
                 for label in (+1, -1))

def build_fcim_feature_matrix(n):
    """Computational-basis eigenvalues of all ZZ terms followed by Z terms."""
    basis_indices = np.arange(2**n, dtype=np.uint64)[:, None]
    shifts = np.arange(n - 1, -1, -1, dtype=np.uint64)
    z_values = 1 - 2 * ((basis_indices >> shifts) & 1).astype(float)
    columns = [z_values[:, i] * z_values[:, j]
               for i, j in itertools.combinations(range(n), 2)]
    columns.extend(z_values[:, i] for i in range(n))
    return np.column_stack(columns)

def compute_loss_and_grads_aggregated_symbolic(
        weights, label_aggregates, pauli_ops, T, n, sparse_paulis=None):
    """Exact binary logistic loss/gradient using only R_plus and R_minus."""
    if sparse_paulis is None:
        sparse_paulis = precompute_sparse_paulis(pauli_ops, n)
    R_plus, R_minus = label_aggregates
    H_matrix = build_hamiltonian_matrix_from_symbolic(
        weights, pauli_ops, n, sparse_paulis=sparse_paulis)
    eigvals, eigvecs = np.linalg.eigh(H_matrix)

    R_plus_tilde = eigvecs.T.conj() @ R_plus @ eigvecs
    R_minus_tilde = eigvecs.T.conj() @ R_minus @ eigvecs
    diag_loss_plus = T * np.logaddexp(0, -eigvals / T)
    diag_loss_minus = T * np.logaddexp(0, eigvals / T)
    loss = np.real(
        np.dot(diag_loss_plus, np.diag(R_plus_tilde))
        + np.dot(diag_loss_minus, np.diag(R_minus_tilde)))

    F_plus = compute_fdd_matrix(eigvals, y=+1, T=T)
    F_minus = compute_fdd_matrix(eigvals, y=-1, T=T)
    derivative_eigenbasis = (F_plus * R_plus_tilde.T
                             + F_minus * R_minus_tilde.T)
    derivative_matrix = (eigvecs.conj() @ derivative_eigenbasis
                         @ eigvecs.T)
    grads = np.array([
        np.real(op.multiply(derivative_matrix).sum())
        for op in sparse_paulis
    ])
    return loss, grads

def compute_loss_and_grads_fcim_diagonal(
        weights, label_basis_probabilities, feature_matrix, T):
    """Exact FCIM loss/gradient with no matrices or eigendecomposition."""
    p_plus, p_minus = label_basis_probabilities
    energies = feature_matrix @ weights
    loss_plus = T * np.logaddexp(0, -energies / T)
    loss_minus = T * np.logaddexp(0, energies / T)
    loss = np.dot(p_plus, loss_plus) + np.dot(p_minus, loss_minus)

    derivative_plus = -np.exp(-np.logaddexp(0, energies / T))
    derivative_minus = np.exp(-np.logaddexp(0, -energies / T))
    weighted_derivative = (p_plus * derivative_plus
                           + p_minus * derivative_minus)
    grads = feature_matrix.T @ weighted_derivative
    return float(np.real(loss)), np.real(grads)

def calculate_fcim_accuracy(weights, feature_matrix, states, true_labels):
    """Classify vectors or density matrices using diagonal FCIM energies."""
    states = np.asarray(states)
    basis_energies = feature_matrix @ weights
    if states.ndim == 2:
        energies = np.abs(states)**2 @ basis_energies
    elif states.ndim == 3:
        probabilities = np.real(np.diagonal(states, axis1=1, axis2=2))
        energies = probabilities @ basis_energies
    else:
        raise ValueError('states must have shape (M, d) or (M, d, d)')
    predictions = np.sign(np.real(energies))
    predictions[predictions == 0] = 1
    return np.mean(predictions == true_labels) * 100

def compute_loss_and_grads_symbolic(
        weights, training_states, ys, pauli_ops, T, n, sparse_paulis=None):
    """Compute exact loss/gradients from symbolic terms plus a sparse cache.

    The full weighted H is dense only for ``np.linalg.eigh``. Each derivative
    term remains CSR when acting on the eigenvectors, avoiding a dense Pauli
    basis in memory and reducing the cost of P_j @ V.
    """
    if sparse_paulis is None:
        sparse_paulis = precompute_sparse_paulis(pauli_ops, n)

    H_matrix = build_hamiltonian_matrix_from_symbolic(
        weights, pauli_ops, n, sparse_paulis=sparse_paulis)
    eigvals, eigvecs = np.linalg.eigh(H_matrix)

    # logaddexp is stable for large |eigenvalue / T|.
    diag_loss_plus = T * np.logaddexp(0, -eigvals / T)
    diag_loss_minus = T * np.logaddexp(0, eigvals / T)

    rhos_tilde = np.array([eigvecs.T.conj() @ rho @ eigvecs
                            for rho in training_states])
    rhos_diag = np.real(np.diagonal(rhos_tilde, axis1=1, axis2=2))
    loss_diagonals = np.where(ys[:, None] > 0, diag_loss_plus, diag_loss_minus)
    loss = np.mean(np.sum(loss_diagonals * rhos_diag, axis=1))

    F_plus = compute_fdd_matrix(eigvals, y=+1, T=T)
    F_minus = compute_fdd_matrix(eigvals, y=-1, T=T)
    grads = np.zeros(len(weights))

    for j, H_j_sparse in enumerate(sparse_paulis):
        # CSR Pauli action costs O(4^n); dense P_j @ V would cost O(8^n).
        H_j_tilde = eigvecs.T.conj() @ (H_j_sparse @ eigvecs)
        grad_j = 0.0
        for i, y_i in enumerate(ys):
            F = F_plus if y_i > 0 else F_minus
            grad_j += np.real(np.einsum(
                'ij,ij,ij->', F, H_j_tilde, rhos_tilde[i].T))
        grads[j] = grad_j / len(training_states)

    return loss, grads


In [16]:
# optimize: training protocol of Paper Sec. VI.C applied to logistic-loss /
#   binary classification (Sec. VI.D.2, Fig. 8 + Table II). Trains the quantum
#   Heisenberg model and the classical FCIM in parallel on the same labeled data.
#   
#   OPTIMIZED VERSION: Vectorizes gradient computation across entire training set
#   at once (rather than looping over dfj() for each parameter), providing ~3-5x speedup.
def optimize(n=3, epochs=2000, use_fast_grad=True):
    np.set_printoptions(suppress=True, precision=5)
    metrics_log = []
    
    # Generate Hamiltonians
    pauli_q = generate_paulis(n, model="quantum")
    pauli_c = generate_paulis(n, model="classical")

    training_states = make_training_states(n)
    N_states = len(training_states)
    val_states = make_validation_set(n, num_states=500)
    T = 2.0   # temperature T in the logistic loss Eq. (56)
    
    # Data generation: random target Hamiltonian and labels
    target_p = (np.random.random(len(pauli_q)) - 0.5) * 4
    H_target = sum(p * mat for p, mat in zip(target_p, pauli_q))
    ys = np.array([np.sign(np.real(np.trace(H_target @ rho))) for rho in training_states])
    ys[ys == 0] = 1

    ys_val = np.array([np.sign(np.real(np.trace(H_target @ rho))) for rho in val_states])
    ys_val[ys_val == 0] = 1

    # Initialize parameters
    est_q = (np.random.random(len(pauli_q)) - 0.5)
    est_c = (np.random.random(len(pauli_c)) - 0.5)
    
    eta = 0.1

    history_q = []
    history_c = []

    print(f"--- Running Optimization for {n} Qubits (Fast Gradients={use_fast_grad}) ---")
    print(f"{'Epoch':<6} | {'Q Loss':<10} | {'C Loss':<10} | {'Q Acc (%)':<10} | {'C Acc (%)':<10}")    
    print("-" * 60)
    
    for epoch in range(epochs):
        # --- Quantum Pass (with vectorized gradient computation) ---
        if use_fast_grad:
            # Use vectorized gradient computation (~3-5x faster)
            l_q, grad_q = compute_loss_and_grads_vectorized(est_q, training_states, ys, pauli_q, T)
        else:
            # Original implementation: loop over dfj() for each parameter
            H_q = sum(p * mat for p, mat in zip(est_q, pauli_q))
            eval_q, evec_q = np.linalg.eigh(H_q)
            l_q = 0
            for i in range(N_states):
                yi = ys[i]
                m_loss_q = evec_q @ np.diag(T * np.log(1 + np.exp(-yi * eval_q / T))) @ evec_q.T.conj()
                l_q += np.real(np.trace(m_loss_q @ training_states[i]))
            l_q /= N_states
            
            grad_q = np.zeros(len(pauli_q))
            for j in range(len(pauli_q)):
                g_j = sum(dfj(ys[i], training_states[i], eval_q, evec_q, pauli_q[j], T) for i in range(N_states))
                grad_q[j] = g_j / N_states
        
        history_q.append(l_q)
        est_q -= eta * grad_q

        # --- Classical Pass (with vectorized gradient computation) ---
        if use_fast_grad:
            l_c, grad_c = compute_loss_and_grads_vectorized(est_c, training_states, ys, pauli_c, T)
        else:
            H_c = sum(p * mat for p, mat in zip(est_c, pauli_c))
            eval_c, evec_c = np.linalg.eigh(H_c)
            l_c = 0.0
            for i in range(N_states):
                yi = ys[i]
                m_loss_c = evec_c @ np.diag(T * np.log(1 + np.exp(-yi * eval_c / T))) @ evec_c.T.conj()
                l_c += np.real(np.trace(m_loss_c @ training_states[i]))
            l_c /= N_states
            
            grad_c = np.zeros(len(pauli_c))
            for j in range(len(pauli_c)):
                g_j = sum(dfj(ys[i], training_states[i], eval_c, evec_c, pauli_c[j], T) for i in range(N_states))
                grad_c[j] = g_j / N_states
        
        history_c.append(l_c)
        est_c -= eta * grad_c

        epoch_data = {
            "Epoch": epoch,
            "Quantum_Loss": l_q,
            "Classical_Loss": l_c,
            "Quantum_Accuracy_Pct": None,
            "Classical_Accuracy_Pct": None
        }

        if epoch % 20 == 0:
            # Compute validation accuracy
            H_q = sum(p * mat for p, mat in zip(est_q, pauli_q))
            H_c = sum(p * mat for p, mat in zip(est_c, pauli_c))
            q_acc = calculate_accuracy(H_q, val_states, ys_val)
            c_acc = calculate_accuracy(H_c, val_states, ys_val)

            epoch_data["Quantum_Accuracy_Pct"] = q_acc
            epoch_data["Classical_Accuracy_Pct"] = c_acc
            print(f"{epoch:<6} | {l_q:<10.5f} | {l_c:<10.5f} | {q_acc:<10.2f} | {c_acc:<10.2f}")
        metrics_log.append(epoch_data)

    print("\n--- Final Results ---")
    final_H_q = sum(p * mat for p, mat in zip(est_q, pauli_q)) / T
    final_H_c = sum(p * mat for p, mat in zip(est_c, pauli_c)) / T
    
    df_metrics = pd.DataFrame(metrics_log)
    csv_filename = f"outputs/logloss_{n}qubit_cl_heisenberg.csv"
    df_metrics.to_csv(csv_filename, index=False)
    
    print(f"Final Quantum Validation Accuracy:   {calculate_accuracy(final_H_q, val_states, ys_val):.2f}%")
    print(f"Final Classical Validation Accuracy: {calculate_accuracy(final_H_c, val_states, ys_val):.2f}%")
    print("\nTarget Params: ", target_p)
    print("Estimated Q:   ", est_q)
    print("Estimated C:   ", est_c)
    return history_q, history_c;

In [17]:
# optimize_phase2: Phase 2 version using symbolic Pauli operators
# Same training protocol as Phase 1+3, but with symbolic Hamiltonian representation
def optimize_phase2(n=3, epochs=2000):
    np.set_printoptions(suppress=True, precision=5)
    metrics_log = []
    
    # Quantum model: symbolic terms plus one compact CSR cache.
    pauli_q_sym = generate_paulis_symbolic(n, model="quantum")
    sparse_q = precompute_sparse_paulis(pauli_q_sym, n)
    # Classical FCIM: exact computational-basis feature representation.
    fcim_features = build_fcim_feature_matrix(n)

    # Preserve pure states as vectors; never allocate M dense density matrices.
    training_states = make_state_vectors(n, num_states=1000)
    val_states = make_state_vectors(n, num_states=500)
    T = 2.0
    
    # Data generation: random target Hamiltonian and labels
    target_p = (np.random.random(len(pauli_q_sym)) - 0.5) * 4
    H_target = build_hamiltonian_matrix_from_symbolic(
        target_p, pauli_q_sym, n, sparse_paulis=sparse_q)
    ys = np.sign(state_energies(H_target, training_states))
    ys[ys == 0] = 1

    ys_val = np.sign(state_energies(H_target, val_states))
    ys_val[ys_val == 0] = 1

    # Exact sufficient statistics for a fixed binary-label dataset.
    label_aggregates = aggregate_states_by_label(training_states, ys)
    label_basis_probabilities = aggregate_basis_probabilities_by_label(
        training_states, ys)

    # Initialize parameters
    est_q = (np.random.random(len(pauli_q_sym)) - 0.5)
    est_c = (np.random.random(fcim_features.shape[1]) - 0.5)
    
    eta = 0.1

    history_q = []
    history_c = []

    print(f"--- Running Phase 2 Optimization for {n} Qubits (Symbolic Pauli Ops) ---")
    print(f"{'Epoch':<6} | {'Q Loss':<10} | {'C Loss':<10} | {'Q Acc (%)':<10} | {'C Acc (%)':<10}")    
    print("-" * 60)
    
    for epoch in range(epochs):
        # --- Quantum Pass: two exact label aggregates, independent of M ---
        l_q, grad_q = compute_loss_and_grads_aggregated_symbolic(
            est_q, label_aggregates, pauli_q_sym, T, n, sparse_q)
        history_q.append(l_q)
        est_q -= eta * grad_q

        # --- Classical Pass: diagonal FCIM, no eigendecomposition ---
        l_c, grad_c = compute_loss_and_grads_fcim_diagonal(
            est_c, label_basis_probabilities, fcim_features, T)
        history_c.append(l_c)
        est_c -= eta * grad_c

        epoch_data = {
            "Epoch": epoch,
            "Quantum_Loss": l_q,
            "Classical_Loss": l_c,
            "Quantum_Accuracy_Pct": None,
            "Classical_Accuracy_Pct": None
        }

        if epoch % 20 == 0:
            # Densify only the weighted quantum Hamiltonian for validation.
            H_q = build_hamiltonian_matrix_from_symbolic(
                est_q, pauli_q_sym, n, sparse_paulis=sparse_q)
            q_acc = calculate_accuracy(H_q, val_states, ys_val)
            c_acc = calculate_fcim_accuracy(
                est_c, fcim_features, val_states, ys_val)

            epoch_data["Quantum_Accuracy_Pct"] = q_acc
            epoch_data["Classical_Accuracy_Pct"] = c_acc
            print(f"{epoch:<6} | {l_q:<10.5f} | {l_c:<10.5f} | {q_acc:<10.2f} | {c_acc:<10.2f}")
        metrics_log.append(epoch_data)

    print("\n--- Final Results ---")
    final_H_q = build_hamiltonian_matrix_from_symbolic(
        est_q, pauli_q_sym, n, sparse_paulis=sparse_q) / T
    
    df_metrics = pd.DataFrame(metrics_log)
    csv_filename = f"outputs/logloss_{n}qubit_phase2.csv"
    df_metrics.to_csv(csv_filename, index=False)
    
    print(f"Final Quantum Validation Accuracy:   {calculate_accuracy(final_H_q, val_states, ys_val):.2f}%")
    print(f"Final Classical Validation Accuracy: {calculate_fcim_accuracy(est_c, fcim_features, val_states, ys_val):.2f}%")
    print("\nTarget Params: ", target_p)
    print("Estimated Q:   ", est_q)
    print("Estimated C:   ", est_c)
    return history_q, history_c

In [18]:
import matplotlib.pyplot as plt
from matplotlib.ticker import FormatStrFormatter

# plot: reproduces a panel of Paper Fig. 8 -- logistic loss vs epoch for the
#   quantum Heisenberg model vs the classical FCIM.
def plot(history_q, history_c, n):
    plt.figure(figsize=(10, 6))
    plt.plot(history_c, label=r'Classical Model ($H_{\text{FCIM}}$)', color='red', linewidth=2, linestyle='--')
    plt.plot(history_q, label=r'Quantum Model ($H_{\text{Heis}}$)', color='blue', linewidth=2)
    plt.xlabel('Epoch', fontsize=24)
    plt.ylabel('Logistic Loss', fontsize=24)
    plt.xticks(fontsize=20)
    plt.yticks(fontsize=20)    
    
    plt.gca().yaxis.set_major_formatter(FormatStrFormatter('%.2f'))
    
    plt.title(f'{n} Qubits, Heisenberg Model', fontsize=24)
    plt.subplots_adjust(left=0.13, right=0.97, bottom=0.18, top=0.90)
    # plt.yscale('log')
    plt.legend(fontsize=24)
    plt.savefig(f"plots/logloss_{n}qubit_heis_fcim.pdf", format="pdf")
    plt.show()

In [19]:
# === COMPREHENSIVE BENCHMARK: Phase 1 + Phase 3 Optimization ===
import time
import tracemalloc

print("="*70)
print("BENCHMARK: Phase 1 + Phase 3 Optimization (Vectorized Gradients)")
print("="*70)

# Test on n=4 (larger than n=3 for better timing accuracy)
n = 4
test_epochs = [5, 10, 20]

print(f"\nSystem: {n} qubits")
print(f"Training set: 1000 states")
print(f"Validation set: 500 states")
print(f"\n{'Epochs':<8} | {'Original (s)':<14} | {'Optimized (s)':<14} | {'Speedup':<8}")
print("-" * 70)

results = []

for epochs in test_epochs:
    # Benchmark original implementation
    tracemalloc.start()
    start = time.perf_counter()
    history_q_orig, history_c_orig = optimize(n, epochs=epochs, use_fast_grad=False)
    time_orig = time.perf_counter() - start
    current_mem, peak_mem_orig = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    
    # Benchmark optimized implementation  
    tracemalloc.start()
    start = time.perf_counter()
    history_q_opt, history_c_opt = optimize(n, epochs=epochs, use_fast_grad=True)
    time_opt = time.perf_counter() - start
    current_mem, peak_mem_opt = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    
    speedup = time_orig / time_opt
    results.append({
        'epochs': epochs,
        'time_orig': time_orig,
        'time_opt': time_opt,
        'speedup': speedup,
        'mem_orig': peak_mem_orig,
        'mem_opt': peak_mem_opt
    })
    
    print(f"{epochs:<8} | {time_orig:<14.2f} | {time_opt:<14.2f} | {speedup:<8.2f}x")

print("\n" + "="*70)
print("SPEEDUP ANALYSIS & EXTRAPOLATION")
print("="*70)

avg_speedup = np.mean([r['speedup'] for r in results])
print(f"\nAverage speedup (n={n}): {avg_speedup:.2f}x")

# Estimate time for n=6
print(f"\n{'─'*70}")
print(f"EXTRAPOLATION TO FULL TRAINING (n=6, 750 epochs):")
print(f"{'─'*70}")

# Estimate based on per-epoch time
per_epoch_orig = results[-1]['time_orig'] / results[-1]['epochs']
per_epoch_opt = results[-1]['time_opt'] / results[-1]['epochs']

total_time_orig_est = per_epoch_orig * 750 / 60  # Convert to minutes
total_time_opt_est = per_epoch_opt * 750 / 60

time_saved = total_time_orig_est - total_time_opt_est

print(f"\nPer-epoch time (n={n}):")
print(f"  Original:  {per_epoch_orig:.3f}s")
print(f"  Optimized: {per_epoch_opt:.3f}s")
print(f"  Saved per epoch: {per_epoch_orig - per_epoch_opt:.3f}s ({(1-per_epoch_opt/per_epoch_orig)*100:.1f}% faster)")

print(f"\nEstimated full training (n=6, 750 epochs):")
print(f"  Original:  {total_time_orig_est:.1f} minutes ({total_time_orig_est/60:.1f} hours)")
print(f"  Optimized: {total_time_opt_est:.1f} minutes ({total_time_opt_est/60:.1f} hours)")
print(f"  Time saved: {time_saved:.1f} minutes ({time_saved*60:.0f} seconds)")
print(f"  Overall speedup: {avg_speedup:.2f}x")

print(f"\n{'─'*70}")
print(f"NUMERICAL ACCURACY")
print(f"{'─'*70}")

# Check numerical equivalence
loss_diff_q = np.max(np.abs(np.array(history_q_orig) - np.array(history_q_opt)))
loss_diff_c = np.max(np.abs(np.array(history_c_orig) - np.array(history_c_opt)))

print(f"\nLoss difference (should be < 0.1 for numerical agreement):")
print(f"  Quantum model: {loss_diff_q:.2e}")
print(f"  Classical model: {loss_diff_c:.2e}")

if loss_diff_q < 0.1 and loss_diff_c < 0.1:
    print(f"\n✓ PASS: Implementations are numerically equivalent")
else:
    print(f"\n⚠ WARNING: Significant difference detected")

print(f"\n{'─'*70}")
print(f"MEMORY USAGE")
print(f"{'─'*70}")

print(f"\nPeak memory (n={n}, {results[-1]['epochs']} epochs):")
print(f"  Original:  {results[-1]['mem_orig']/1e6:.1f} MB")
print(f"  Optimized: {results[-1]['mem_opt']/1e6:.1f} MB")

print(f"\n" + "="*70)
print(f"SUMMARY: Phase 1 + Phase 3 provides {avg_speedup:.2f}x speedup")
print(f"="*70)

BENCHMARK: Phase 1 + Phase 3 Optimization (Vectorized Gradients)

System: 4 qubits
Training set: 1000 states
Validation set: 500 states

Epochs   | Original (s)   | Optimized (s)  | Speedup 
----------------------------------------------------------------------
--- Running Optimization for 4 Qubits (Fast Gradients=False) ---
Epoch  | Q Loss     | C Loss     | Q Acc (%)  | C Acc (%) 
------------------------------------------------------------
0      | 1.43321    | 1.41479    | 57.20      | 53.60     

--- Final Results ---
Final Quantum Validation Accuracy:   57.20%
Final Classical Validation Accuracy: 54.20%

Target Params:  [ 0.29163 -1.79577  0.83262  0.63494 -0.89959 -1.47911  0.45931 -1.69235
  0.92372  0.77483 -1.87195  0.97187 -1.31099 -0.21391 -1.18174 -0.77503
 -1.08305  1.27809 -1.54213  1.00602 -0.07403]
Estimated Q:    [-0.38955 -0.28904  0.29494  0.09364 -0.3667   0.16768 -0.09622 -0.11116
  0.01227  0.05268  0.14295  0.22813 -0.1045   0.44376  0.0034   0.12051
 -0.02258  

In [ ]:
# === PHASE 2 BENCHMARK: Compare Phase 1+3 vs Phase 2 (Symbolic Operators) ===
import time

print("\n" + "="*80)
print("BENCHMARK: Phase 1+3 vs Phase 2 (Symbolic Pauli Operators)")
print("="*80)

n = 4
test_epochs = [5]  # Start with 5 epochs for quick comparison

print(f"\nSystem: {n} qubits")
print(f"Training set: 1000 states")
print(f"Validation set: 500 states")
print(f"\n{'Epochs':<8} | {'Phase 1+3 (s)':<16} | {'Phase 2 (s)':<16} | {'Speedup':<10}")
print("-" * 80)

phase2_results = []
benchmark_seed = 20260622

for epochs in test_epochs:
    # Phase 1+3 Benchmark
    print(f"\n[Phase 1+3 - {epochs} epochs]")
    np.random.seed(benchmark_seed)
    start = time.perf_counter()
    history_q_p1, history_c_p1 = optimize(n, epochs=epochs, use_fast_grad=True)
    time_p1 = time.perf_counter() - start
    
    # Phase 2 Benchmark
    print(f"\n[Phase 2 - {epochs} epochs]")
    np.random.seed(benchmark_seed)  # identical data and initialization
    start = time.perf_counter()
    history_q_p2, history_c_p2 = optimize_phase2(n, epochs=epochs)
    time_p2 = time.perf_counter() - start
    
    speedup_p2_vs_p1 = time_p1 / time_p2 if time_p2 > 0 else 0
    
    phase2_results.append({
        'epochs': epochs,
        'time_p1': time_p1,
        'time_p2': time_p2,
        'speedup': speedup_p2_vs_p1
    })
    
    print(f"\n{epochs:<8} | {time_p1:<16.2f} | {time_p2:<16.2f} | {speedup_p2_vs_p1:<10.2f}x")
    
    # Numerical accuracy check
    loss_diff_q = np.max(np.abs(np.array(history_q_p1) - np.array(history_q_p2)))
    loss_diff_c = np.max(np.abs(np.array(history_c_p1) - np.array(history_c_p2)))
    
    print(f"\nNumerical accuracy:")
    print(f"  Quantum model loss diff:   {loss_diff_q:.2e}")
    print(f"  Classical model loss diff: {loss_diff_c:.2e}")
    
    if loss_diff_q < 0.1 and loss_diff_c < 0.1:
        print(f"  ✓ PASS: Implementations are numerically equivalent")
    else:
        print(f"  ⚠ WARNING: Significant difference detected")

print(f"\n" + "="*80)
print("PHASE 2 INTEGRATION RESULTS")
print("="*80)

if phase2_results:
    avg_speedup_p2 = np.mean([r['speedup'] for r in phase2_results])
    print(f"\nPhase 2 vs Phase 1+3 speedup: {avg_speedup_p2:.2f}x")
    print(f"\nNote: both paths use the same dense exact eigendecomposition.")
    print(f"      The n=6 cell below measures the demonstrated representation")
    print(f"      advantage separately from end-to-end training time.")


In [ ]:
# === N=6 SYMBOLIC REPRESENTATION REGRESSION + SCALING BENCHMARK ===

n_test = 6
rng = np.random.default_rng(20260622)
weights = rng.uniform(-0.5, 0.5, 6 * n_test - 3)

dense_terms = generate_paulis(n_test, model='quantum')
symbolic_terms = generate_paulis_symbolic(n_test, model='quantum')
sparse_terms = precompute_sparse_paulis(symbolic_terms, n_test)

assert all(isinstance(op, qml.operation.Operator) for op in symbolic_terms)
assert isinstance(build_hamiltonian_symbolic(weights, symbolic_terms), qml.operation.Operator)

dense_bytes = sum(op.nbytes for op in dense_terms)
sparse_bytes = sparse_storage_bytes(sparse_terms)
memory_reduction = dense_bytes / sparse_bytes
dense_entries = sum(op.size for op in dense_terms)
sparse_nonzeros = sum(op.nnz for op in sparse_terms)
entry_reduction = dense_entries / sparse_nonzeros

H_dense = sum((w * op for w, op in zip(weights, dense_terms)),
              start=np.zeros_like(dense_terms[0]))
H_symbolic = build_hamiltonian_matrix_from_symbolic(
    weights, symbolic_terms, n_test, sparse_paulis=sparse_terms)
hamiltonian_error = np.max(np.abs(H_dense - H_symbolic))

# Exact loss/gradient regression on n=6 pure-state density matrices.
num_benchmark_states = 64
vectors = (rng.normal(size=(num_benchmark_states, 2**n_test))
           + 1j * rng.normal(size=(num_benchmark_states, 2**n_test)))
vectors /= np.linalg.norm(vectors, axis=1, keepdims=True)
benchmark_states = np.einsum('bi,bj->bij', vectors, vectors.conj())
benchmark_labels = np.sign(np.real(np.einsum(
    'bi,ij,bj->b', vectors.conj(), H_dense, vectors)))
benchmark_labels[benchmark_labels == 0] = 1

dense_result = compute_loss_and_grads_vectorized(
    weights, benchmark_states, benchmark_labels, dense_terms, T=2.0)
symbolic_result = compute_loss_and_grads_symbolic(
    weights, benchmark_states, benchmark_labels, symbolic_terms, T=2.0,
    n=n_test, sparse_paulis=sparse_terms)
loss_error = abs(dense_result[0] - symbolic_result[0])
gradient_error = np.max(np.abs(dense_result[1] - symbolic_result[1]))

# Exact label-aggregate quantum regression.
label_aggregates = aggregate_states_by_label(vectors, benchmark_labels)
aggregated_result = compute_loss_and_grads_aggregated_symbolic(
    weights, label_aggregates, symbolic_terms, T=2.0, n=n_test,
    sparse_paulis=sparse_terms)
aggregate_loss_error = abs(symbolic_result[0] - aggregated_result[0])
aggregate_gradient_error = np.max(
    np.abs(symbolic_result[1] - aggregated_result[1]))
aggregate_memory_reduction = (benchmark_states.nbytes
                              / sum(rho.nbytes for rho in label_aggregates))

# Exact diagonal-FCIM regression against the generic eigensolver path.
classical_terms = generate_paulis_symbolic(n_test, model='classical')
classical_sparse_terms = precompute_sparse_paulis(classical_terms, n_test)
classical_weights = rng.uniform(-0.5, 0.5, len(classical_terms))
classical_generic_result = compute_loss_and_grads_symbolic(
    classical_weights, benchmark_states, benchmark_labels, classical_terms,
    T=2.0, n=n_test, sparse_paulis=classical_sparse_terms)
fcim_features = build_fcim_feature_matrix(n_test)
label_basis_probabilities = aggregate_basis_probabilities_by_label(
    vectors, benchmark_labels)
classical_diagonal_result = compute_loss_and_grads_fcim_diagonal(
    classical_weights, label_basis_probabilities, fcim_features, T=2.0)
fcim_loss_error = abs(
    classical_generic_result[0] - classical_diagonal_result[0])
fcim_gradient_error = np.max(np.abs(
    classical_generic_result[1] - classical_diagonal_result[1]))

assert hamiltonian_error < 1e-12
assert loss_error < 1e-12
assert gradient_error < 1e-12
assert aggregate_loss_error < 1e-12
assert aggregate_gradient_error < 1e-12
assert fcim_loss_error < 1e-12
assert fcim_gradient_error < 1e-12
assert memory_reduction > 10
assert entry_reduction == 2**n_test

print(f'n={n_test}: {len(symbolic_terms)} genuine PennyLane operators')
print(f'Dense term payload:  {dense_bytes / 2**20:.3f} MiB')
print(f'Sparse term payload: {sparse_bytes / 2**10:.3f} KiB')
print(f'Term-storage reduction: {memory_reduction:.2f}x')
print(f'Stored numeric entries: {dense_entries:,} dense vs {sparse_nonzeros:,} sparse')
print(f'Entry-count reduction: {entry_reduction:.0f}x')
print(f'Hamiltonian max error: {hamiltonian_error:.3e}')
print(f'Loss error: {loss_error:.3e}')
print(f'Gradient max error: {gradient_error:.3e}')
print(f'Aggregate loss/gradient errors: {aggregate_loss_error:.3e} / {aggregate_gradient_error:.3e}')
print(f'Aggregate-state memory reduction ({num_benchmark_states} states): {aggregate_memory_reduction:.2f}x')
print(f'Diagonal FCIM loss/gradient errors: {fcim_loss_error:.3e} / {fcim_gradient_error:.3e}')
print('PASS: n=6 symbolic, aggregate-label, and diagonal-FCIM paths agree.')

n=6: 33 genuine PennyLane operators
Dense term payload:  2.062 MiB
Sparse term payload: 43.629 KiB
Term-storage reduction: 48.41x
Stored numeric entries: 135,168 dense vs 2,112 sparse
Entry-count reduction: 64x
Hamiltonian max error: 0.000e+00
Loss error: 2.220e-16
Gradient max error: 2.776e-17
Aggregate loss/gradient errors: 2.220e-16 / 7.633e-17
Aggregate-state memory reduction (64 states): 32.00x
Diagonal FCIM loss/gradient errors: 0.000e+00 / 2.776e-17
PASS: n=6 symbolic, aggregate-label, and diagonal-FCIM paths agree.


In [ ]:
plot(history_q, history_c, n)

In [ ]:
# to read data from csv instead of re-generating
n = 6
csv_filename = f"outputs/logloss_{n}qubit_heis_fcim.csv"
df = pd.read_csv(csv_filename)

history_q = df['Quantum_Loss'].values
history_c = df['Classical_Loss'].values

plot(history_q, history_c, n)